# Cache Aside 패턴과 Pipeline

Redis를 애플리케이션 캐시로 활용하는 **Cache Aside 패턴**과  
여러 명령을 묶어 처리하는 **Pipeline**을 배웁니다.

In [1]:
import os
import json
import time
import redis
import pandas as pd
from dotenv import load_dotenv

# .env 파일에서 환경변수를 불러옵니다
load_dotenv()

# Redis 클라이언트 생성 (decode_responses=True로 bytes → str 자동 변환)
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    db=int(os.getenv('REDIS_DB', 0)),
    password=os.getenv('REDIS_PASSWORD') or None,
    decode_responses=True,
)

# True가 출력되면 Redis 서버에 정상 연결된 것입니다
print(r.ping())  # True

True


## 1. Cache Aside 패턴

가장 일반적인 캐시 사용 패턴입니다.

```
요청 → Redis에서 읽기
  ├─ 캐시 히트(Hit) : Redis에서 바로 반환
  └─ 캐시 미스(Miss): DB 조회 → Redis에 저장 → 반환
```

| 상황 | 처리 | 속도 |
|---|---|---|
| 캐시 히트 | Redis에서 즉시 반환 | 빠름 |
| 캐시 미스 | DB 조회 후 Redis에 저장 | 느림 (최초 1회) |

In [2]:
def db_query_product(product_id: int) -> dict:
    """실제 서비스에서는 RDB를 조회합니다. (0.1초 지연 시뮬레이션)"""
    time.sleep(0.1)   # DB 조회에 걸리는 시간을 흉내냅니다
    return {'id': product_id, 'name': f'상품 {product_id}', 'price': product_id * 1000}


In [3]:
def get_product(product_id: int, ttl: int = 60) -> dict:
    # 캐시 키는 'cache:product:{id}' 형태로 통일합니다
    cache_key = f'cache:product:{product_id}'

    # 1단계: Redis에서 먼저 조회 (캐시 히트 여부 확인)
    cached = r.get(cache_key)

    if cached:
        # 캐시 히트 — Redis에 저장된 JSON 문자열을 dict로 변환해 즉시 반환합니다
        print(f'[캐시 히트] {cache_key}')
        return json.loads(cached)

    # 캐시 미스 — Redis에 데이터가 없으므로 DB를 조회합니다
    print(f'[캐시 미스] {cache_key}')
    product = db_query_product(product_id)

    # 2단계: 조회 결과를 Redis에 저장해 다음 요청부터 빠르게 반환합니다
    # json.dumps: dict → JSON 문자열 변환 (Redis는 문자열만 저장 가능)
    # ensure_ascii=False: 한글이 \uXXXX 코드로 깨지지 않도록 합니다
    r.set(cache_key, json.dumps(product, ensure_ascii=False), ex=ttl)
    return product

In [4]:
# 첫 번째 요청 — 캐시 미스 (DB 조회 발생, 약 100ms 소요)
# r.delete로 캐시를 지워 항상 미스 상태에서 시작합니다
r.delete('cache:product:42')

start = time.time()
product = get_product(42)
print(f'결과: {product}')
print(f'응답 시간: {(time.time() - start) * 1000:.1f}ms')  # 약 100ms

[캐시 미스] cache:product:42
결과: {'id': 42, 'name': '상품 42', 'price': 42000}
응답 시간: 105.3ms


In [5]:
# 두 번째 요청 — 캐시 히트 (Redis에서 즉시 반환, 1ms 미만)
# 첫 번째 요청이 Redis에 저장해 두었으므로 DB 조회 없이 바로 응답합니다
start = time.time()
product = get_product(42)
print(f'결과: {product}')
print(f'응답 시간: {(time.time() - start) * 1000:.1f}ms')  # < 1ms

[캐시 히트] cache:product:42
결과: {'id': 42, 'name': '상품 42', 'price': 42000}
응답 시간: 2.0ms


In [6]:
# 히트 / 미스 응답 시간 비교 — 5번 요청해서 첫 번째(미스)와 나머지(히트)를 표로 비교합니다
r.delete('cache:product:99')   # 캐시 초기화로 첫 요청이 반드시 미스가 되도록 합니다

records = []
for i in range(5):
    start = time.time()
    get_product(99)
    elapsed_ms = (time.time() - start) * 1000
    # i == 0이면 캐시가 없어 DB 조회(미스), 이후는 모두 캐시에서 응답(히트)
    status = '캐시 미스' if i == 0 else '캐시 히트'
    records.append({'요청 번호': i + 1, '상태': status, '응답시간(ms)': round(elapsed_ms, 2)})

pd.DataFrame(records)

[캐시 미스] cache:product:99
[캐시 히트] cache:product:99
[캐시 히트] cache:product:99
[캐시 히트] cache:product:99
[캐시 히트] cache:product:99


,요청 번호,상태,응답시간(ms)
0,1,캐시 미스,102.53
1,2,캐시 히트,0.57
2,3,캐시 히트,0.53
3,4,캐시 히트,0.52
4,5,캐시 히트,0.55


## 2. Pipeline

여러 명령을 한 번에 묶어 서버로 보냅니다. 네트워크 왕복 횟수를 줄여 성능이 향상됩니다.

```
일반    : 명령1 → 응답1, 명령2 → 응답2, 명령3 → 응답3  (왕복 3번)
Pipeline: [명령1, 명령2, 명령3] → [응답1, 응답2, 응답3]  (왕복 1번)
```

> 순서가 보장되며 중간에 오류가 나도 나머지 명령은 실행됩니다.  
> 원자적 처리가 필요하면 `r.pipeline(transaction=True)`를 사용합니다.

In [7]:
# 게시글 5개의 조회수를 Pipeline으로 일괄 조회
article_ids = [101, 102, 103, 104, 105]

# 초기값 설정 (각 게시글의 조회수를 article_id * 10으로 가정)
for aid in article_ids:
    r.set(f'views:article:{aid}', aid * 10)

# Pipeline 사용 — r.pipeline()으로 파이프라인 객체를 생성합니다
# pipe.get(...)은 즉시 실행되지 않고, execute() 호출 시 한 번에 서버로 전송됩니다
pipe = r.pipeline()
for aid in article_ids:
    pipe.get(f'views:article:{aid}')
views = pipe.execute()   # [결과1, 결과2, ...] 리스트로 반환됩니다

df = pd.DataFrame({'게시글': [f'article:{aid}' for aid in article_ids], '조회수': views})
df

,게시글,조회수
0,article:101,1010
1,article:102,1020
2,article:103,1030
3,article:104,1040
4,article:105,1050


In [8]:
# 일반 방식 vs Pipeline 성능 비교 — 명령 수가 많을수록 차이가 커집니다
n = 200
keys = [f'bench:{i}' for i in range(n)]

# 일반 방식: 명령마다 서버 왕복(Round Trip)이 발생합니다 → 네트워크 지연이 n번 누적
start = time.time()
for k in keys:
    r.set(k, 1)
normal_ms = (time.time() - start) * 1000

# Pipeline 방식: 명령을 묶어 한 번에 전송 → 네트워크 왕복이 1~2번으로 줄어듭니다
start = time.time()
pipe = r.pipeline()
for k in keys:
    pipe.set(k, 1)
pipe.execute()   # 여기서 한 번에 서버로 전송됩니다
pipeline_ms = (time.time() - start) * 1000

# 사용한 키 정리 (다음 실행 시 오염 방지)
pipe = r.pipeline()
for k in keys:
    pipe.delete(k)
pipe.execute()

pd.DataFrame([
    {'방식': '일반', '명령 수': n, '소요시간(ms)': round(normal_ms, 1)},
    {'방식': 'Pipeline', '명령 수': n, '소요시간(ms)': round(pipeline_ms, 1)},
])

,방식,명령 수,소요시간(ms)
0,일반,200,172.6
1,Pipeline,200,1.7


## 3. ConnectionPool

웹 서버처럼 동시 요청이 많은 환경에서는 매 요청마다 새 연결을 만들면 비용이 큽니다.  
**ConnectionPool**로 연결을 미리 만들어 재사용합니다.

| 방식 | 설명 |
|---|---|
| 연결을 매번 생성 | 연결 비용 발생, 동시 처리 어려움 |
| ConnectionPool | 연결 재사용, 동시 요청 안정적으로 처리 |

In [9]:
# ConnectionPool — 연결을 미리 만들어 재사용합니다
# 웹 서버는 동시에 수백 개의 요청을 처리하므로, 매 요청마다 새 연결을 맺으면 비용이 큽니다
# max_connections: 풀에서 동시에 사용 가능한 최대 연결 수
pool = redis.ConnectionPool(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    db=int(os.getenv('REDIS_DB', 0)),
    decode_responses=True,
    max_connections=10,   # 동시에 최대 10개 연결을 재사용
)

# connection_pool 인자로 풀을 연결한 클라이언트를 생성합니다
# 이 클라이언트는 명령을 실행할 때 풀에서 연결을 빌려 쓰고, 완료되면 반납합니다
r_pool = redis.Redis(connection_pool=pool)
print(r_pool.ping())  # True
print(f'최대 연결 수: {pool.max_connections}')

True
최대 연결 수: 10


## 실습

1. 상품 3개(`product:1`, `product:2`, `product:3`)를 DB에서 조회하는 함수를 만들고,  
   Cache Aside 패턴을 적용하세요. 첫 번째와 두 번째 조회의 응답 시간을 비교하세요.
2. Pipeline으로 사용자 10명의 포인트(`points:user:1` ~ `points:user:10`)를 한 번에 저장하고,  
   결과를 DataFrame으로 출력하세요.
3. 위 포인트를 일반 방식과 Pipeline 방식으로 각각 읽어 소요 시간을 비교하세요.

### 1. 상품 3개에 Cache Aside 패턴 적용 및 응답 시간 비교

In [10]:
def db_query_ex(product_id: int) -> dict:
    time.sleep(0.1)   # DB 조회 지연 시뮬레이션
    return {'id': product_id, 'name': f'실습상품 {product_id}', 'price': product_id * 5000}

def get_product_ex(product_id: int, ttl: int = 60) -> dict:
    cache_key = f'ex:cache:product:{product_id}'
    cached = r.get(cache_key)
    if cached:
        # 캐시 히트: JSON 문자열 → dict 변환 후 반환
        return json.loads(cached), '캐시 히트'
    # 캐시 미스: DB 조회 후 Redis에 저장
    product = db_query_ex(product_id)
    r.set(cache_key, json.dumps(product, ensure_ascii=False), ex=ttl)
    return product, '캐시 미스'

In [11]:
# 캐시 초기화 — 모든 상품이 미스 상태에서 시작하도록 합니다
for pid in [1, 2, 3]:
    r.delete(f'ex:cache:product:{pid}')

In [12]:
# 각 상품을 2번씩 요청합니다: 첫 번째는 미스(DB 조회), 두 번째는 히트(Redis 반환)
records = []
for pid in [1, 2, 3, 1, 2, 3]:
    start = time.time()
    _, status = get_product_ex(pid)
    elapsed_ms = (time.time() - start) * 1000
    records.append({'product_id': pid, '상태': status, '응답시간(ms)': round(elapsed_ms, 2)})

pd.DataFrame(records)

,product_id,상태,응답시간(ms)
0,1,캐시 미스,103.76
1,2,캐시 미스,104.88
2,3,캐시 미스,105.38
3,1,캐시 히트,1.65
4,2,캐시 히트,1.00
5,3,캐시 히트,1.00


### 2. Pipeline으로 사용자 10명 포인트 일괄 저장 후 DataFrame 출력

In [13]:
# 저장: 파이프라인으로 10개의 set 명령을 한 번에 전송합니다
pipe = r.pipeline()
for i in range(1, 11):
    pipe.set(f'points:user:{i}', i * 100)   # user:1=100, user:2=200, ...
    
pipe.execute()

[True, True, True, True, True, True, True, True, True, True]

In [14]:
# 조회: 저장과 마찬가지로 파이프라인으로 10개 키를 한 번에 조회합니다
# execute()는 명령 순서대로 결과 리스트를 반환합니다
pipe = r.pipeline()
for i in range(1, 11):
    pipe.get(f'points:user:{i}')
points = pipe.execute()   # ['100', '200', '300', ..., '1000']

df = pd.DataFrame({
    '사용자': [f'user:{i}' for i in range(1, 11)],
    '포인트': [int(p) for p in points],   # 문자열로 온 값을 int로 변환
})

df.head()

,사용자,포인트
0,user:1,100
1,user:2,200
2,user:3,300
3,user:4,400
4,user:5,500


In [15]:
df.tail()

,사용자,포인트
5,user:6,600
6,user:7,700
7,user:8,800
8,user:9,900
9,user:10,1000


### 3. 일반 방식 vs Pipeline 읽기 시간 비교 (위에서 저장한 포인트 사용)

In [ ]:
# 일반 방식: 10번의 get 명령이 각각 서버를 왕복합니다
start = time.time()

for i in range(1, 11):
    r.get(f'points:user:{i}')

normal_ms = (time.time() - start) * 1000

In [17]:
# Pipeline 방식: 10개의 get 명령을 묶어 한 번에 전송합니다
start = time.time()
pipe = r.pipeline()

for i in range(1, 11):
    pipe.get(f'points:user:{i}')

pipe.execute()
pipeline_ms = (time.time() - start) * 1000

In [18]:
# 사용한 키 정리 — 파이프라인으로 일괄 삭제해 다음 실행에 영향을 주지 않도록 합니다
pipe = r.pipeline()
for i in range(1, 11):
    pipe.delete(f'points:user:{i}')
pipe.execute()

pd.DataFrame([
    {'방식': '일반',     '명령 수': 10, '소요시간(ms)': round(normal_ms, 2)},
    {'방식': 'Pipeline', '명령 수': 10, '소요시간(ms)': round(pipeline_ms, 2)},
])

,방식,명령 수,소요시간(ms)
0,일반,10,3.17
1,Pipeline,10,0.57
